<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/4_model_training/4_5_model_mlp.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4_5_model_mlp

## Introducción y Resumen

El objetivo de esta notebook es generar las ventanas de entrada (X) y targets (y) para los horizontes de 30, 60 y 90 minutos, aplica el escalado de features sobre cada conjunto (train, valid, test) y finalmente guarda las ventanas escaladas en disco, dejándolas listas para el entrenamiento de modelos.

0. Configuración del Entorno

    Se conecta Google Drive y se clona el repositorio de trabajo. Se instalan e importan librerías necesarias como pandas, numpy, matplotlib y seaborn. Se cargan los datasets procesados previamente y se muestra un resumen de la información de los datasets.

1. Carga de datos

    Se importan los datasets procesados `mnq_train`, `mnq_valid` y `mnq_test`. Se revisa su estructura (filas, columnas, tipos de datos) y también de importa el listado de features seleccionados para cada ventana de tiempo.





## 0. Configuración del Entorno


### 0.1. Instalación de librerías


In [1]:
# Alineamos el stack numérico
%pip install --no-cache-dir -U \
  numpy==2.1.2 \
  scipy==1.13.1 \
  scikit-learn==1.5.2

IndentationError: unexpected indent (ipython-input-2999694949.py, line 3)

### 0.2. Importación de librerías


In [2]:
# ==============================
# Librerías estándar de Python
# ==============================
import os
import sys
import re
import glob
import warnings
import requests
from datetime import datetime, timedelta
from functools import reduce

# ==============================
# Manejo y procesamiento de datos
# ==============================
import pandas as pd
import numpy as np
from tabulate import tabulate

# ==============================
# Visualización
# ==============================
import matplotlib.pyplot as plt

# ==============================
# Estadística
# ==============================
from scipy.stats import spearmanr

# ==============================
# Machine Learning y utilidades
# ==============================
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from sklearn.neural_network import MLPRegressor
import sklearn, scipy, numpy #, optuna

import joblib

# ==============================
# Configuración general
# ==============================
warnings.filterwarnings("ignore")

import time

import sys, platform, lightgbm as lgb
import numpy as np, pandas as pd
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score


In [3]:
print("python:", sys.version)
print("Platform:", platform.platform())
print("numpy:", numpy.__version__)
print("scipy:", scipy.__version__)
print("sklearn:", sklearn.__version__)
#print("optuna:", optuna.__version__)
#print("xgboost:", xgb.__version__)
#print("lightgbm:", lgb.__version__)

# CatBoost usa la clase para exponer versión
#print("catboost:", catboost.__version__)

python: 3.12.11 (main, Jun  4 2025, 08:56:18) [GCC 11.4.0]
Platform: Linux-6.6.97+-x86_64-with-glibc2.35
numpy: 2.0.2
scipy: 1.16.2
sklearn: 1.6.1


### 0.3. Acceso a Drive

In [4]:
from google.colab import drive
drive.mount('/content/drive')
drive_path = "/content/drive/MyDrive/neural_profit"

Mounted at /content/drive


## 1. Carga de datos

### 1.1. Carga de datasets `mnq_train`, `mnq_valid` y `mnq_test`






In [5]:
def load_data(data: str):

    data_path = f'{drive_path}/3_dataset_preparation/mnq_{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)

    # Asegurar que el índice esté en formato datetime
    df.index = pd.to_datetime(df.index)

    # Crear una nueva columna 'date' con la fecha extraída del índice
    df['date'] = df.index.date

    # Reordenar columnas: 'date', 'time_str', y luego el resto
    cols = ['date'] + [col for col in df.columns if col not in ['date']]

    df = df[cols]

    return df

In [6]:
#mnq_train = load_data("train")
#mnq_valid = load_data("valid")
#mnq_test = load_data("test")

### 1.2. Información de datasets


In [7]:
def info_dataset(df, name: str):
  print(f"Información del dataset {name}:\n")

  # Contar valores únicos en la columna 'date'
  num_dias = df['date'].nunique()
  print(f"\tCantidad de días: {num_dias}")

  # Filtrar valores válidos
  validos_por_dia = df.dropna(subset=['close']).groupby('date').size()

  # Calcular el promedio
  promedio_por_fecha = validos_por_dia.mean()
  print(f"\tRegistros por día: {int(promedio_por_fecha)}")

  primer_hora = df.index[0].strftime('%H:%M')
  ultima_hora = df.index[-1].strftime('%H:%M')
  zona_horaria = df.index[0].tzinfo


  print(f"\tHora diaria de inicio {primer_hora}")
  print(f"\tHora diaria de final {ultima_hora}")
  print(f"\tZona horaria: {zona_horaria}\n")

  return num_dias, promedio_por_fecha

In [8]:
#info_dataset(mnq_train, 'mnq_train')
#info_dataset(mnq_valid, 'mnq_valid')
#info_dataset(mnq_test, 'mnq_test')

### 1.3. Carga de listado de features por ventana de tiempo

In [9]:
import json

# Ruta al archivo guardado
path = f'{drive_path}/2_feature_engineering/features_list.json'

with open(path, "r") as f:
    features_dict = json.load(f)

# Extraer las listas
#features_to_30 = features_dict["features_to_30"]
#features_to_60 = features_dict["features_to_60"]
#features_to_90 = features_dict["features_to_90"]


In [10]:
#print(f'Listado de features para 30min: {features_to_30}')
#print(f'Listado de features para 60min: {features_to_60}')
#print(f'Listado de features para 90min: {features_to_90}')

## 2. Carga de ventanas `X_train_*_scaled`, `X_valid_*_scaled`, `X_test_*_scaled`

### 2.0. Funciones

#### Función para cargar ventanas

In [11]:
def load_windows_and_scaler(target: str, scaled=True):
    """
    Carga datasets (X, y) para train, valid y test junto con el scaler global.

    Parámetros
    ----------
    drive_path : str
        Ruta base donde se encuentran los archivos.
    scaled : bool, default=True
        Si True busca en la carpeta 'ventanas_x_y_scaled',
        si False en 'ventanas_x_y'.

    Retorna
    -------
    X_train, y_train, X_valid, y_valid, X_test, y_test, scaler
    """

    #Ruta de ventandas escaladas
    path_train  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_train_{target}_scaled.npz'
    path_valid  = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_valid_{target}_scaled.npz'
    path_test   = f'{drive_path}/3_dataset_preparation/xy_windows_scaled/xy_test_{target}_scaled.npz'

    #Ruta de escalador
    path_scaler = f"{drive_path}/3_dataset_preparation/global_scaler_{target}.pkl"

    # Cargar npz
    data_train = np.load(path_train)
    data_valid = np.load(path_valid)
    data_test  = np.load(path_test)

    # Extraer X, y
    X_train, y_train = data_train["X"], data_train["y"]
    X_valid, y_valid = data_valid["X"], data_valid["y"]
    X_test,  y_test  = data_test["X"],  data_test["y"]

    # Cargar scaler
    scaler = joblib.load(path_scaler)

    return X_train, y_train, X_valid, y_valid, X_test, y_test, scaler


#### Función para revisar información de ventanas

In [12]:
def xy_info(target: str, X_train, y_train, X_valid, y_valid, X_test, y_test):
    print(f'Información para horizonte de {target} minutos:')

    for name, X, y in [
        ("entrenamiento", X_train, y_train),
        ("validación", X_valid, y_valid),
        ("testeo", X_test, y_test),
    ]:
        print(f'\nSet de {name}:')
        print(f'\t{X.shape[0]} ventanas (n_samples).')

        if X.ndim == 2:

            print(f'\t{X.shape[1]} features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_{target})')
        elif X.ndim == 3:

            print(f'\t{X.shape[1]} pasos en lookback × {X.shape[2]} features por paso. Dimensión 3D: (n_samples, window_size, len(features_{target})')

        print(f'\t{y.shape[0]} targets.')
        print(f'\tDistribución y: mean={y.mean():.6f}, std={y.std():.6f}, min={y.min():.6f}, max={y.max():.6f}')

    return  X_train.shape[0], X_valid.shape[0], X_test.shape[0]

### 2.1 Carga de ventanas 30 minutos

In [13]:
X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30, scaler_30 = load_windows_and_scaler(target = '30')

In [14]:
n_samples_train_30, n_samples_valid_30, n_samples_test_30 = xy_info( '30', X_train_30_scaled, y_train_30, X_valid_30_scaled, y_valid_30, X_test_30_scaled, y_test_30)

Información para horizonte de 30 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	193487 targets.
	Distribución y: mean=0.000056, std=0.002760, min=-0.028931, max=0.032372

Set de validación:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=0.000103, std=0.003273, min=-0.023167, max=0.068056

Set de testeo:
	41567 ventanas (n_samples).
	900 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_30)
	41567 targets.
	Distribución y: mean=-0.000023, std=0.002407, min=-0.015982, max=0.016097


### 2.2 Carga de ventanas 60 minutos

In [15]:
X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60, scaler_60 = load_windows_and_scaler(target = '60')

In [16]:
n_samples_train_60, n_samples_valid_60, n_samples_test_60 = xy_info( '60', X_train_60_scaled, y_train_60, X_valid_60_scaled, y_valid_60, X_test_60_scaled, y_test_60)

Información para horizonte de 60 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	193487 targets.
	Distribución y: mean=0.000114, std=0.003907, min=-0.040179, max=0.036224

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=0.000163, std=0.004643, min=-0.038084, max=0.079896

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_60)
	41567 targets.
	Distribución y: mean=-0.000068, std=0.003521, min=-0.018494, max=0.020365


### 2.3 Carga de ventanas 90 minutos

In [17]:
X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90, scaler_90 = load_windows_and_scaler(target = '90')

In [18]:
n_samples_train_90, n_samples_valid_90, n_samples_test_90 = xy_info('90', X_train_90_scaled, y_train_90, X_valid_90_scaled, y_valid_90, X_test_90_scaled, y_test_90)

Información para horizonte de 90 minutos:

Set de entrenamiento:
	193487 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	193487 targets.
	Distribución y: mean=0.000159, std=0.004746, min=-0.037742, max=0.039284

Set de validación:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=0.000224, std=0.005887, min=-0.040748, max=0.083184

Set de testeo:
	41567 ventanas (n_samples).
	1080 features totales por ventana (aplanado). Dimensión 2D: (n_samples, window_size * len(features_90)
	41567 targets.
	Distribución y: mean=-0.000132, std=0.004558, min=-0.023392, max=0.026213


## 3. Dataset de Métricas

Dado que cada entrenamiento demanda un tiempo considerable, antes de proceder verificaremos si ya existe un resultado previo de este modelo consultando el dataset de métricas.

### 3.1. Función para cargar métricas o generar dataset

In [19]:
def load_metrics(data: str):
    data_path = f'{drive_path}/4_model_training/{data}.parquet'
    # Leer el archivo Parquet y cargarlo en un DataFrame
    df = pd.read_parquet(data_path)
    return df

In [20]:
def metrics_verify(data: str) -> bool:
    data_path = f"{drive_path}/4_model_training/{data}.parquet"
    return os.path.exists(data_path)


In [21]:
def load_or_create_metrics (data:str):
  if metrics_verify(data):
      print(f"Las métricas existen y son almacenadas en {data[4:len(data)]}")
      model_metrics = load_metrics(data)
      #print(random_forest_metrics)
      metrics = True
  else:
      print(f"Las métricas no existen. Se crea el dataset {data[4:len(data)]} para almacenar las métricas")
      #Creamos la tabla para almacenar las métricas
      model_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])
      metrics = False

  return model_metrics, metrics

In [22]:
mlp_metrics, metrics = load_or_create_metrics("4_5_mlp_metrics")

Las métricas no existen. Se crea el dataset mlp_metrics para almacenar las métricas


### 3.2. Función para guardar métricas

In [23]:
def save_metrics (metrics,  metrics_name: str):   #("4_2_xgboost_metrics")
  metrics_path = f"{drive_path}/4_model_training/{metrics_name}.parquet"
  metrics.to_parquet(metrics_path, index = True)
  print(f"Métricas guardadas en {metrics_path}")

### 3.3. Función para calcular las métricas

In [26]:
def evaluate_model(model, X, y_true, y_pred=None, eps=1e-8):
    """
    Evalúa RMSE, MAE, R2, SMAPE y DirAcc.
    - Si y_pred es None, predice con el modelo usando X.
    - Evita mean_squared_error(squared=...) para máxima compatibilidad.
    """
    if y_pred is None:
        y_pred = model.predict(X)

    # Asegurar 1D
    y_true = np.ravel(y_true)
    y_pred = np.ravel(y_pred)

    # RMSE sin sklearn
    rmse = float(np.sqrt(np.mean((y_true - y_pred) ** 2)))
    mae = float(mean_absolute_error(y_true, y_pred))
    r2  = float(r2_score(y_true, y_pred))

    # SMAPE
    smape_val = 100.0 * np.mean(
        (np.abs(y_true - y_pred) / ((np.abs(y_true) + np.abs(y_pred)) / 2.0 + eps))
    )

    # Directional Accuracy
    directional_acc = float(np.mean(np.sign(y_true) == np.sign(y_pred)))

    return {
        "RMSE": rmse,
        "MAE": mae,
        "R2": r2,
        "SMAPE": float(smape_val),
        "DirAcc": directional_acc
    }

In [27]:
def print_metrics(metrics, target:str):
  print(f"Métricas de {target}:\n")
  for k, v in metrics.items():
      print(f"\t{k:>5}:\t {float(v):.6f}")

## 4. Definición de modelo


### 4.1. Función de entrenamiento para modelo

In [28]:
from typing import Tuple, Optional, Dict, Any
import numpy as np
from sklearn.neural_network import MLPRegressor
from sklearn.preprocessing import StandardScaler

In [29]:
def train_model_mlp(
    best_params: Optional[Dict[str, Any]],
    X_train: np.ndarray, y_train: np.ndarray,
    X_valid: np.ndarray, y_valid: np.ndarray,
    *,
    use_internal_early_stopping: bool = True,
    hidden_layer_sizes: Tuple[int, ...] = (128, 64),
    activation: str = "relu",
    solver: str = "adam",
    alpha: float = 1e-4,
    learning_rate: str = "adaptive",
    max_iter: int = 500,
    n_iter_no_change: int = 25,
    random_state: int = 42,
    verbose: bool = False
):
    """
    Entrena un MLPRegressor y devuelve (modelo, preds_valid).
    X_train/X_valid ya deben venir escalados si corresponde.
    """

    # 1) Defaults + overrides del usuario
    params = dict(best_params or {})
    params.setdefault("hidden_layer_sizes", hidden_layer_sizes)
    params.setdefault("activation", activation)
    params.setdefault("solver", solver)
    params.setdefault("alpha", alpha)
    params.setdefault("learning_rate", learning_rate)
    params.setdefault("max_iter", max_iter)
    params.setdefault("random_state", random_state)
    params.setdefault("verbose", verbose)
    params.setdefault("shuffle", False)  # default temporal

    # 2) Evitar duplicados con lo que fija la función
    for k in ("early_stopping", "validation_fraction", "n_iter_no_change"):
        params.pop(k, None)

    # 3) Construcción y entrenamiento
    if use_internal_early_stopping:
        X_all = np.vstack([X_train, X_valid])
        y_all = np.concatenate([y_train, y_valid])
        val_frac = len(X_valid) / float(len(X_all))

        model = MLPRegressor(
            **params,
            early_stopping=True,
            n_iter_no_change=n_iter_no_change,
            validation_fraction=val_frac
        )
        model.fit(X_all, y_all)
    else:
        model = MLPRegressor(
            **params,
            early_stopping=False
        )
        model.fit(X_train, y_train)

    preds_valid = model.predict(X_valid)
    return model, preds_valid

### 4.2. Parámetros por defecto para modelo


In [30]:
mlp_default_params = {
    "hidden_layer_sizes": (128, 64),   # arquitectura base: 2 capas ocultas
    "activation": "relu",              # función de activación: 'relu', 'tanh', 'logistic', 'identity'
    "solver": "adam",                  # optimizador: 'adam', 'lbfgs', 'sgd'
    "alpha": 1e-4,                     # regularización L2
    "batch_size": "auto",              # tamaño de mini-lote (auto = min(200, n_samples))
    "learning_rate": "adaptive",       # ajusta la tasa según mejora del loss
    "learning_rate_init": 0.001,       # tasa inicial de aprendizaje
    "power_t": 0.5,                    # solo para 'sgd'; exponente de schedule del learning rate
    "max_iter": 500,                   # iteraciones máximas
    #"shuffle": False,                  # mantenemos orden temporal
    "shuffle": True,                  # mantenemos orden temporal

    "random_state": 42,                # reproducibilidad
    "tol": 1e-4,                       # tolerancia para criterio de convergencia
    "momentum": 0.9,                   # solo para 'sgd'
    "nesterovs_momentum": True,        # solo para 'sgd'
    #"early_stopping": True,            # habilitamos early stopping interno
    #"n_iter_no_change": 25,            # paciencia para early stopping
    #"validation_fraction": 0.1,        # fracción de validación interna
    "beta_1": 0.9,                     # coeficiente beta1 del Adam
    "beta_2": 0.999,                   # coeficiente beta2 del Adam
    "epsilon": 1e-8,                   # estabilidad numérica de Adam
    "verbose": False,                  # logs del entrenamiento
    "warm_start": False,               # False = reinicia pesos cada vez
    "max_fun": 15000,                  # límite interno de optimización (solo lbfgs)
}

### 4.3. Función conjunta

In [31]:
# Usa la función que ya te pasé antes:
# from your_module import train_model_mlp

def _internal_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    """Fallback si no pasás evaluate_fn. Calcula RMSE, MAE, R2, SMAPE y DirAcc."""
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    mae  = float(mean_absolute_error(y_true, y_pred))
    r2   = float(r2_score(y_true, y_pred))
    # SMAPE clásico (en %) con epsilon para evitar div/0
    eps = 1e-12
    denom = (np.abs(y_true) + np.abs(y_pred)).clip(min=eps)
    smape = float(np.mean(2.0 * np.abs(y_pred - y_true) / denom) * 100.0)
    diracc = float(np.mean(np.sign(y_pred) == np.sign(y_true)))
    return {"RMSE": rmse, "MAE": mae, "R2": r2, "SMAPE": smape, "DirAcc": diracc}

def run_mlp_experiment(
    metrics_flag: bool,
    model_key: str,
    X_train_scaled: np.ndarray,
    y_train: np.ndarray,
    X_valid_scaled: np.ndarray,
    y_valid: np.ndarray,
    resample_rate: float,                 # 0.3, 0.5, 1.0
    mlp_params: Dict[str, Any],
    mlp_metrics_df: Optional[pd.DataFrame] = None,
    n_samples_train: Optional[int] = None,
    n_samples_valid: Optional[int] = None,
    *,
    # extras específicos de MLP / sklearn
    use_internal_early_stopping: bool = True,
    max_iter: int = 500,
    n_iter_no_change: int = 25,
    verbose: bool = False,
    # hooks opcionales
    evaluate_fn=None,          # si tenés tu evaluate_model(model, X, y, y_pred=...) pasalo acá
    print_metrics_fn=None      # si tenés tu print_metrics(dict, model_key) pasalo acá
) -> dict:
    """
    Ejecuta un experimento MLP con (opcional) subsampleo de train/valid.

    Parámetros
    ----------
    metrics_flag : bool
        Si True, no entrena y busca métricas previas en mlp_metrics_df[model_key].
    model_key : str
        Nombre/índice del modelo en la tabla de métricas (ej: 'MLP_60_subsampleado_50%').
    X_train_scaled, y_train : arrays
        Ventanas y target de entrenamiento (ya escaladas previamente).
    X_valid_scaled, y_valid : arrays
        Ventanas y target de validación (ya escaladas previamente).
    resample_rate : float
        Proporción a muestrear (0 < r <= 1). 1.0 = sin subsampleo.
    mlp_params : dict
        Hiperparámetros nativos del MLPRegressor (hidden_layer_sizes, alpha, etc.).
    mlp_metrics_df : pd.DataFrame | None
        DataFrame de métricas para leer/escribir (index por model_key). Opcional.
    n_samples_train, n_samples_valid : int | None
        Tamaños base para calcular la cantidad a muestrear. Si es None, se infiere de X_*.

    Retorna
    -------
    dict
        Diccionario con métricas {"RMSE","MAE","R2","SMAPE","DirAcc"}.
    """
    # 1) Carga de métricas previas (si corresponde)
    if metrics_flag and (mlp_metrics_df is not None) and (model_key in mlp_metrics_df.index):
        print("El modelo fue entrenado anteriormente y las métricas ya fueron calculadas\n")
        metrics_dict = mlp_metrics_df.loc[model_key].to_dict()
        if print_metrics_fn is not None:
            print_metrics_fn(metrics_dict, model_key)
        else:
            print(f"[{model_key}] -> {metrics_dict}")
        return metrics_dict

    # 2) Mensaje de entrenamiento
    tag_rate = f"{int(resample_rate*100)}%" if resample_rate < 1.0 else "100%"
    print(f"Entrenando modelo {model_key} (resample={tag_rate})...")

    # 3) Tamaños base
    if n_samples_train is None:
        n_samples_train = X_train_scaled.shape[0]
    if n_samples_valid is None:
        n_samples_valid = X_valid_scaled.shape[0]

    # 4) Subsampleo (si aplica)
    if resample_rate < 1.0:
        n_train_sub = max(1, int(n_samples_train * resample_rate))
        n_valid_sub = max(1, int(n_samples_valid * resample_rate))
        X_train_sub, y_train_sub = subsample(X_train_scaled, y_train, n_train_sub)
        X_valid_sub, y_valid_sub = subsample(X_valid_scaled, y_valid, n_valid_sub)
    else:
        X_train_sub, y_train_sub = X_train_scaled, y_train
        X_valid_sub, y_valid_sub = X_valid_scaled, y_valid

    # 5) Entrenamiento y predicción en valid
    model, y_pred_valid = train_model_mlp(
        best_params=mlp_params,
        X_train=X_train_sub, y_train=y_train_sub,
        X_valid=X_valid_sub, y_valid=y_valid_sub,
        use_internal_early_stopping=use_internal_early_stopping,
        max_iter=max_iter,
        n_iter_no_change=n_iter_no_change,
        verbose=verbose
    )

    # 6) Evaluación (usa tu evaluate_model si lo pasaste; si no, usa fallback interno)
    if evaluate_fn is not None:
        metrics_dict = evaluate_fn(model, X_valid_sub, y_valid_sub, y_pred=y_pred_valid)
    else:
        metrics_dict = _internal_metrics(y_valid_sub, y_pred_valid)

    # 7) Mostrar
    if print_metrics_fn is not None:
        print_metrics_fn(metrics_dict, model_key)
    else:
        print(f"[{model_key}] -> {metrics_dict}")

    # 8) Persistir métricas en el DataFrame si se pasa
    if mlp_metrics_df is not None:
        mlp_metrics_df.loc[model_key] = metrics_dict

    return metrics_dict

### 4.4. Función para subsamplear

In [32]:
def subsample(X, y, n):
    n = min(n, X.shape[0])
    idx = np.random.choice(X.shape[0], size=n, replace=False)
    return X[idx], y[idx]

In [33]:
#xgb_device_test = xgb.XGBRegressor(tree_method="gpu_hist", predictor="gpu_predictor")
#try:
#    xgb_device_test.fit([[0,0],[1,1]], [0,1])
#    print("✅ XGBoost GPU works correctly")
#except Exception as e:
#    print("❌ GPU not available for XGBoost:", e)

## 5. Entrenamiento

### 5.1. Entrenamiento 30min

#### 5.1.1. Con 30% de dataset

In [50]:
mlp_30_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_subsampleado_30%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time 188s / 3,13min

Entrenando modelo MLP_30_subsampleado_30% (resample=30%)...
Métricas de MLP_30_subsampleado_30%:

	 RMSE:	 0.002537
	  MAE:	 0.001655
	   R2:	 0.333589
	SMAPE:	 120.453559
	DirAcc:	 0.671852


In [34]:
mlp_metrics

,RMSE,MAE,R2,SMAPE,DirAcc


#### 5.1.2. Con 50% de dataset

In [35]:
mlp_30_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_subsampleado_50%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Tiempo: 774s / 13min

Entrenando modelo MLP_30_subsampleado_50% (resample=50%)...
Métricas de MLP_30_subsampleado_50%:

	 RMSE:	 0.002543
	  MAE:	 0.001607
	   R2:	 0.397152
	SMAPE:	 119.067749
	DirAcc:	 0.699514


#### 5.1.3. Con ventanas completas

In [ ]:
'''
mlp_30_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_30_100%",
    X_train_scaled=X_train_30_scaled, y_train=y_train_30,
    X_valid_scaled=X_valid_30_scaled, y_valid=y_valid_30,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Tiempo:
'''

Entrenando modelo CAT_30_100% (resample=100%)...
0:	learn: 0.0027481	test: 0.0032623	best: 0.0032623 (0)	total: 71.7ms	remaining: 5m 58s
200:	learn: 0.0021864	test: 0.0028549	best: 0.0028549 (200)	total: 8.38s	remaining: 3m 20s
400:	learn: 0.0020683	test: 0.0028195	best: 0.0028195 (400)	total: 15.6s	remaining: 2m 58s
600:	learn: 0.0019880	test: 0.0028067	best: 0.0028067 (600)	total: 24s	remaining: 2m 55s
800:	learn: 0.0019222	test: 0.0027992	best: 0.0027992 (799)	total: 31.8s	remaining: 2m 46s
1000:	learn: 0.0018676	test: 0.0027934	best: 0.0027934 (999)	total: 40.2s	remaining: 2m 40s
1200:	learn: 0.0018186	test: 0.0027884	best: 0.0027883 (1198)	total: 49s	remaining: 2m 35s
1400:	learn: 0.0017740	test: 0.0027845	best: 0.0027844 (1399)	total: 56.6s	remaining: 2m 25s
1600:	learn: 0.0017342	test: 0.0027819	best: 0.0027819 (1600)	total: 1m 5s	remaining: 2m 18s
1800:	learn: 0.0016974	test: 0.0027803	best: 0.0027802 (1789)	total: 1m 13s	remaining: 2m 10s
2000:	learn: 0.0016626	test: 0.0027799

### 4.2. Entrenamiento 60min

#### 4.2.1. Con 30% de dataset

In [36]:
mlp_60_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_subsampleado_30%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 390s /6.5m

Entrenando modelo MLP_60_subsampleado_30% (resample=30%)...
Métricas de MLP_60_subsampleado_30%:

	 RMSE:	 0.003266
	  MAE:	 0.001943
	   R2:	 0.505579
	SMAPE:	 101.422228
	DirAcc:	 0.767442


#### 4.2.2. Con 50% de dataset

In [37]:
mlp_60_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_subsampleado_50%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time: 484 seg

Entrenando modelo MLP_60_subsampleado_50% (resample=50%)...
Métricas de MLP_60_subsampleado_50%:

	 RMSE:	 0.003372
	  MAE:	 0.001955
	   R2:	 0.453291
	SMAPE:	 101.994321
	DirAcc:	 0.767887


#### 4.2.3. Con ventanas completas

In [39]:
'''
mlp_60_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_60_100%",
    X_train_scaled=X_train_60_scaled, y_train=y_train_60,
    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time
'''

'\nmlp_60_100 = run_mlp_experiment(\n    metrics_flag=metrics,\n    model_key="MLP_60_100%",\n    X_train_scaled=X_train_60_scaled, y_train=y_train_60,\n    X_valid_scaled=X_valid_60_scaled, y_valid=y_valid_60,\n    resample_rate=1.0,\n    mlp_params=mlp_default_params,\n    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí\n    use_internal_early_stopping=True,\n    max_iter=600,                           # podés subirlo un poco\n    n_iter_no_change=30,\n    verbose=False,\n    # Si ya tenés estas funciones genéricas, podés pasarlas:\n    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF\n    print_metrics_fn=print_metrics\n)\n#Time \n'

### 4.3. Entrenamiento 90min

#### 4.3.1. Con 30% de dataset

In [ ]:
mlp_90_30 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_subsampleado_30%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.3,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time:

Entrenando modelo MLP_90_subsampleado_30% (resample=30%)...


#### 4.3.2. Con 50% de dataset

In [ ]:
mlp_90_50 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_subsampleado_50%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=0.5,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time:

Entrenando modelo CAT_90_subsampleado_50% (resample=50%)...
0:	learn: 0.0047367	test: 0.0058823	best: 0.0058823 (0)	total: 101ms	remaining: 8m 25s
200:	learn: 0.0031356	test: 0.0047426	best: 0.0047426 (200)	total: 8.72s	remaining: 3m 28s
400:	learn: 0.0028530	test: 0.0046140	best: 0.0046140 (400)	total: 17.7s	remaining: 3m 22s
600:	learn: 0.0026952	test: 0.0045646	best: 0.0045646 (600)	total: 25.9s	remaining: 3m 9s
800:	learn: 0.0025748	test: 0.0045267	best: 0.0045265 (798)	total: 34.4s	remaining: 3m
1000:	learn: 0.0024731	test: 0.0044988	best: 0.0044988 (1000)	total: 43.5s	remaining: 2m 53s
1200:	learn: 0.0023881	test: 0.0044779	best: 0.0044777 (1197)	total: 52.2s	remaining: 2m 45s
1400:	learn: 0.0023134	test: 0.0044604	best: 0.0044604 (1400)	total: 1m	remaining: 2m 36s
1600:	learn: 0.0022438	test: 0.0044476	best: 0.0044471 (1589)	total: 1m 9s	remaining: 2m 28s
1800:	learn: 0.0021818	test: 0.0044365	best: 0.0044364 (1796)	total: 1m 18s	remaining: 2m 18s
2000:	learn: 0.0021225	test: 0.

#### 4.3.3. Con ventanas completas

In [ ]:
'''
mlp_90_100 = run_mlp_experiment(
    metrics_flag=metrics,
    model_key="MLP_90_100%",
    X_train_scaled=X_train_90_scaled, y_train=y_train_90,
    X_valid_scaled=X_valid_90_scaled, y_valid=y_valid_90,
    resample_rate=1.0,
    mlp_params=mlp_default_params,
    mlp_metrics_df=mlp_metrics,            # guarda/lee métricas aquí
    use_internal_early_stopping=True,
    max_iter=600,                           # podés subirlo un poco
    n_iter_no_change=30,
    verbose=False,
    # Si ya tenés estas funciones genéricas, podés pasarlas:
    evaluate_fn=evaluate_model,             # mismo que usás con CAT/LGBM/XGB/RF
    print_metrics_fn=print_metrics
)
#Time:
'''

Entrenando modelo CAT_90_100% (resample=100%)...
0:	learn: 0.0047147	test: 0.0058604	best: 0.0058604 (0)	total: 82.2ms	remaining: 6m 50s
200:	learn: 0.0031383	test: 0.0047210	best: 0.0047210 (200)	total: 9.95s	remaining: 3m 57s
400:	learn: 0.0028641	test: 0.0046026	best: 0.0046026 (400)	total: 20.1s	remaining: 3m 50s
600:	learn: 0.0027063	test: 0.0045522	best: 0.0045522 (600)	total: 29.2s	remaining: 3m 33s
800:	learn: 0.0025858	test: 0.0045147	best: 0.0045147 (800)	total: 39.4s	remaining: 3m 26s
1000:	learn: 0.0024862	test: 0.0044845	best: 0.0044845 (1000)	total: 49.6s	remaining: 3m 18s
1200:	learn: 0.0024005	test: 0.0044592	best: 0.0044591 (1199)	total: 59.9s	remaining: 3m 9s
1400:	learn: 0.0023264	test: 0.0044431	best: 0.0044430 (1398)	total: 1m 9s	remaining: 2m 57s
1600:	learn: 0.0022577	test: 0.0044260	best: 0.0044258 (1598)	total: 1m 19s	remaining: 2m 48s
1800:	learn: 0.0021983	test: 0.0044117	best: 0.0044116 (1797)	total: 1m 29s	remaining: 2m 38s
2000:	learn: 0.0021423	test: 0.00

## 5. Recuperación de métricas

In [ ]:
mlp_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
CAT_30_subsampleado_30%,0.002918,0.001633,0.260324,114.789048,0.709222
CAT_30_subsampleado_50%,0.002641,0.001601,0.291650,114.807198,0.710100
CAT_60_subsampleado_30%,0.003507,0.001895,0.413397,94.783952,0.789014
CAT_60_subsampleado_50%,0.003509,0.001876,0.406737,95.195925,0.786220
CAT_90_subsampleado_30%,0.004337,0.002156,0.457987,87.852341,0.813713
CAT_90_subsampleado_50%,0.004366,0.002137,0.453938,86.784671,0.817351
CAT_30_100%,0.002775,0.001599,0.280774,114.050754,0.713378
CAT_60_100%,0.003576,0.001891,0.406758,95.194200,0.786249
CAT_90_100%,0.004334,0.002135,0.458094,86.951187,0.816633


In [ ]:
save_metrics(mlp_metrics, "4_4_catboost_metrics")

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_4_catboost_metrics.parquet


Este punto existe para garantizar reproducibilidad y continuidad del análisis sin reentrenar modelos cuando se pierden las métricas. Actúa como fallback: reconstruye la tabla de métricas de Random Forest a partir de valores ya validados y la persiste nuevamente, evitando el re-entrenamiento de los modelos.

Así, se mantiene la consistencia de resultados y la trazabilidad de comparaciones y conclusiones, incluso si el archivo original fue eliminado, corrompido o el entorno de ejecución cambió.

In [ ]:
def generate_metrics_mlp():
  '''
  Ejecutar está función solo en caso de perder las métricas
  El objetivo es no volver a correr los entrenamientos
  '''
  mlp_metrics = pd.DataFrame(columns=["RMSE", "MAE", "R2", "SMAPE", "DirAcc"])

  mlp_30_30 = {
      "RMSE":	 0.002537,
      "MAE":	 0.001655,
      "R2":	 0.333589,
      "SMAPE":	 120.453559,
      "DirAcc":	 0.671852
  }

  mlp_30_50 = {
      "RMSE": 0.002543,
      "MAE": 0.001607,
      "R2": 0.397152,
      "SMAPE": 119.067749,
      "DirAcc": 0.699514
  }

  mlp_60_30 = {
      "RMSE": 0.003266,
      "MAE": 0.001943,
      "R2": 0.505579,
      "SMAPE": 101.422228,
      "DirAcc": 0.767442
  }

  mlp_60_50 = {
    'RMSE': 0.003372370312304722,
    'MAE': 0.001954590335922684,
    'R2': 0.4532909519346531,
    'SMAPE': 101.99432110111918,
    'DirAcc': 0.7678872155126786
 }
############
  mlp_90_30 = {
      "RMSE": 0.004383,
      "MAE": 0.002170,
      "R2": 0.450930,
      "SMAPE": 87.363224,
      "DirAcc": 0.812991
  }


  mlp_90_50 = {
      "RMSE": 0.004452,
      "MAE":  0.002165,
      "R2": 0.448462,
      "SMAPE": 87.058502,
      "DirAcc": 0.817688
      }

  mlp_metrics.loc['MLP_30_subsampleado_30%'] = mlp_30_30
  mlp_metrics.loc['MLP_30_subsampleado_50%'] = mlp_30_50

  mlp_metrics.loc['MLP_60_subsampleado_30%'] = mlp_60_30
  mlp_metrics.loc['MLP_60_subsampleado_50%'] = mlp_60_50

  mlp_metrics.loc['MLP_90_subsampleado_30%'] = mlp_90_30
  mlp_metrics.loc['MLP_90_subsampleado_50%'] = mlp_90_50


  save_metrics(mlp_metrics, "4_5_mlp_metrics")
  return mlp_metrics

In [ ]:
#Ejecutar esta función solo en caso de perder las métricas de entrenamiento
#mlp_metrics = generate_metrics_mlp()

Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet


In [ ]:
mlp_metrics

,RMSE,MAE,R2,SMAPE,DirAcc
LGBM_30_subsampleado_30%,0.002875,0.001621,0.268676,114.153659,0.709463
LGBM_30_subsampleado_50%,0.002726,0.001601,0.287471,114.231020,0.711495
LGBM_60_subsampleado_30%,0.003493,0.001892,0.395036,95.774103,0.781957
LGBM_60_subsampleado_50%,0.003598,0.001905,0.405742,96.080063,0.785594
LGBM_90_subsampleado_30%,0.004383,0.002170,0.450930,87.363224,0.812991
LGBM_90_subsampleado_50%,0.004452,0.002165,0.448462,87.058502,0.817688


Métricas guardadas en /content/drive/MyDrive/neural_profit/4_model_training/4_3_lgbm_metrics.parquet
